# Koi Breed Classifier — Colab Training

Trains the timm-based koi breed classifier on a GPU runtime, then writes the model + a report back to Google Drive.

**Flow:** clone repo → mount Drive → copy & unzip dataset → train → export ONNX → save model + report to Drive.

## Before you run
1. **Runtime → Change runtime type → GPU** (T4 is fine).
2. **Dataset**: upload your dataset **as a zip** to Google Drive (e.g. `MyDrive/koi-breed/data.zip`). Inside, one folder per breed with images.
3. Edit the **Settings** cell below if your paths differ, then **Runtime → Run all**.

## 1. Settings — edit these

In [ ]:
# ===== GitHub (public repo) =====
REPO_URL = "https://github.com/wfxuraz/koi-class.git"   # full clone URL
BRANCH   = "main"                                       # branch to train from

# ===== Google Drive layout (under MyDrive) =====
DRIVE_DATA_ZIP   = "/content/drive/MyDrive/koi-breed/data.zip"   # your zipped dataset
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/koi-breed/runs"       # where model + report get written

# ===== Local (fast SSD) working paths — usually leave as-is =====
LOCAL_DATA  = "/content/data"   # dataset is unzipped here
LOCAL_RUNS  = "/content/runs"   # training writes here, then we copy to Drive
DATA_SUBDIR = ""                # subfolder inside the zip holding breed dirs; "" = auto-detect

## 2. Clone the repo

In [ ]:
import os

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.exists(REPO_DIR):
    !rm -rf "$REPO_DIR"

!git clone --branch $BRANCH --depth 1 $REPO_URL $REPO_DIR
os.chdir(REPO_DIR)
print("working dir:", os.getcwd())

## 3. Install dependencies (GPU-safe)

Colab already ships a CUDA-enabled `torch`. We install everything **except** torch/torchvision so we don't downgrade to the CPU build and lose the GPU.

In [ ]:
!pip -q install "timm>=1.0.7" "onnx>=1.16" onnxruntime onnxscript "scikit-learn>=1.4" "pyyaml>=6.0" "pillow>=10.0"
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 4. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Copy & unzip the dataset

Copy the zip to the local SSD first — unzipping and training from `/content` is far faster than reading from Drive directly.

In [ ]:
import os, zipfile, shutil

assert os.path.exists(DRIVE_DATA_ZIP), f"zip not found in Drive: {DRIVE_DATA_ZIP}"
os.makedirs(LOCAL_DATA, exist_ok=True)
local_zip = "/content/_data.zip"

print("copying zip from Drive ...")
shutil.copy(DRIVE_DATA_ZIP, local_zip)
print("unzipping ...")
with zipfile.ZipFile(local_zip) as z:
    z.extractall(LOCAL_DATA)
print("done")

## 6. Locate data, drop non-breed dirs, point config at it

In [ ]:
import os, yaml

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def find_class_root(base):
    """Return the directory whose subfolders look most like breed/image dirs."""
    best, best_score = base, -1
    for root, dirs, _ in os.walk(base):
        score = 0
        for d in dirs:
            if d.startswith('.'):
                continue
            p = os.path.join(root, d)
            if any(f.lower().endswith(IMG_EXTS) for f in os.listdir(p)):
                score += 1
        if score > best_score:
            best, best_score = root, score
    return best

data_dir = os.path.join(LOCAL_DATA, DATA_SUBDIR) if DATA_SUBDIR else find_class_root(LOCAL_DATA)
print("data_dir:", data_dir)

# drop the '_bk' bookkeeping folder flagged in the data report (it has no images)
bk = os.path.join(data_dir, "_bk")
if os.path.isdir(bk):
    shutil.move(bk, "/content/_bk")
    print("moved _bk out of the data dir")

# show per-class counts
classes = sorted(d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d)))
print(f"\n{len(classes)} classes:")
total = 0
for c in classes:
    n = sum(1 for f in os.listdir(os.path.join(data_dir, c)) if f.lower().endswith(IMG_EXTS))
    total += n
    print(f"  {c:20s} {n}")
print(f"total images: {total}")

# rewrite config.yaml to point at the unzipped data and the local runs dir
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["data_dir"] = data_dir
cfg["output_dir"] = LOCAL_RUNS
with open("config.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
print("\nconfig.yaml updated")

## 7. Train

Per-epoch macro-F1 / accuracy / lowest-recall classes stream below and are also saved to `train.log` for the report.

In [ ]:
!python -m koi.train --config config.yaml 2>&1 | tee /content/train.log

## 8. Export ONNX (for CPU/edge deployment)

In [ ]:
!python -m koi.export_onnx --config config.yaml

## 9. Write model + report to Google Drive

In [ ]:
import os, shutil, datetime, yaml

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
data_dir = cfg["data_dir"]
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
classes = sorted(d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d)))
counts = {c: sum(1 for f in os.listdir(os.path.join(data_dir, c)) if f.lower().endswith(IMG_EXTS))
          for c in classes}
log = open("/content/train.log").read() if os.path.exists("/content/train.log") else ""

lines = [
    "# Koi Breed Classifier — Training Report", "",
    f"- Date: {stamp}",
    f"- Repo / branch: {REPO_URL} @ {BRANCH}",
    f"- Backbone: {cfg['backbone']}  |  image_size: {cfg['image_size']}",
    f"- Epochs: {cfg['epochs']}  |  batch_size: {cfg['batch_size']}  |  lr: {cfg['lr']}  |  cb_beta: {cfg['cb_beta']}",
    "", "## Class counts", "", "| Breed | Count |", "|---|---:|",
]
for c in sorted(counts, key=lambda k: -counts[k]):
    lines.append(f"| {c} | {counts[c]} |")
lines += [
    "", f"Total images: {sum(counts.values())}  |  Classes: {len(classes)}", "",
    "## Training log", "", "```", log.strip(), "```", "",
]
report = "\n".join(lines)
with open("/content/report.md", "w") as f:
    f.write(report)

# copy artifacts to Drive
saved = []
for name in ["best_model.pt", "classes.json", "model.onnx"]:
    src = os.path.join(LOCAL_RUNS, name)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(DRIVE_OUTPUT_DIR, name))
        saved.append(name)
for name, src in [("report.md", "/content/report.md"), ("train.log", "/content/train.log")]:
    if os.path.exists(src):
        shutil.copy(src, os.path.join(DRIVE_OUTPUT_DIR, name))
        saved.append(name)

print("written to", DRIVE_OUTPUT_DIR)
for s in saved:
    print("  ✓", s)
print("\n" + report[:1000])